In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
import os
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =======================================================================================
#
# BLOCK 3: LOAD ALL PRE-TRAINED MODEL PREDICTIONS
#
# =======================================================================================
PREDS_PATH = './model_predictions/'

print("Loading all base model predictions from saved .npy files...")

try:
    # --- Load XGBoost Predictions ---
    # The original script named this oof_mean_preds. We load it and rename for clarity.
    oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy') 
    test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
    print("  - XGBoost predictions loaded.")

    # --- Load CatBoost Predictions ---
    oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy')
    print("  - CatBoost predictions loaded.")

    # --- Load Neural Network Predictions ---
    oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
    print("  - Neural Network predictions loaded.")
    
except FileNotFoundError as e:
    print(f"\nERROR: Could not find a prediction file. {e}")
    print("Please ensure you have run all three training notebooks and saved their predictions first.")
    # You might want to exit() here if a file is not found

# Load the target variable needed for calculating RMSE
y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
print("\nAll predictions loaded successfully.")

# --- OOF RMSE of Loaded Models for verification ---
print("\n--- OOF RMSE of All Base Models ---")
print(f"XGBoost  : ${np.sqrt(mean_squared_error(y_true, oof_xgb_preds)):,.2f}")
print(f"CatBoost : ${np.sqrt(mean_squared_error(y_true, oof_cb_preds)):,.2f}")
print(f"NeuralNet: ${np.sqrt(mean_squared_error(y_true, oof_nn_preds)):,.2f}")

Loading all base model predictions from saved .npy files...
  - XGBoost predictions loaded.
  - CatBoost predictions loaded.
  - Neural Network predictions loaded.

All predictions loaded successfully.

--- OOF RMSE of All Base Models ---
XGBoost  : $98,990.27
CatBoost : $98,246.71
NeuralNet: $108,435.86


In [6]:
# =======================================================================================
#
# BLOCK 4: FIND THE OPTIMAL 3-MODEL BLEND & CREATE PREDICTIONS
#
# =======================================================================================
from scipy.optimize import minimize

print("\n--- Searching for the optimal 3-model blend weights ---")

# Stack the OOF predictions into a single array for the optimizer
oof_preds_stack = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T

def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

initial_weights = [1/3, 1/3, 1/3]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - np.sum(w)})
bounds = [(0, 1)] * len(initial_weights)

result = minimize(get_ensemble_rmse, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)

best_weights = result.x
best_ensemble_rmse = result.fun

print("\n" + "="*50)
print("             OPTIMAL 3-MODEL BLEND SEARCH COMPLETE")
print("="*50)
print(f"Lowest Ensemble RMSE achieved: ${best_ensemble_rmse:,.2f}")
print(f"Optimal Weight for XGBoost  : {best_weights[0]:.4f}")
print(f"Optimal Weight for CatBoost : {best_weights[1]:.4f}")
print(f"Optimal Weight for NeuralNet: {best_weights[2]:.4f}")
print("="*50)


# --- NEW PART C: Create the Super-Ensemble Mean Predictions ---
# This part was missing. It creates the variables needed by the next cell.
print("\n--- Creating the super-ensemble mean predictions using these optimal weights ---")

oof_ensemble_mean = (oof_xgb_preds * best_weights[0] + 
                     oof_cb_preds * best_weights[1] + 
                     oof_nn_preds * best_weights[2])

test_ensemble_mean = (test_xgb_preds * best_weights[0] + 
                      test_cb_preds * best_weights[1] + 
                      test_nn_preds * best_weights[2])

print("Super-ensemble mean predictions are now created and ready for use.")


--- Searching for the optimal 3-model blend weights ---

             OPTIMAL 3-MODEL BLEND SEARCH COMPLETE
Lowest Ensemble RMSE achieved: $96,778.10
Optimal Weight for XGBoost  : 0.4100
Optimal Weight for CatBoost : 0.4779
Optimal Weight for NeuralNet: 0.1121

--- Creating the super-ensemble mean predictions using these optimal weights ---
Super-ensemble mean predictions are now created and ready for use.


In [7]:
# =======================================================================================
#
# BLOCK 5: TRAIN & SAVE THE CHAMPION XGBOOST ERROR MODEL
#
# =======================================================================================

# --- PART A: Create the data for the error models ---
# This assumes you have already created the 3-model oof_ensemble_mean and test_ensemble_mean
print("--- Preparing data for error model training (using 3-model super-ensemble) ---")
error_target_ensemble = np.abs(y_true - oof_ensemble_mean)
X_for_error_ensemble = X.copy()
X_for_error_ensemble['mean_pred_oof'] = oof_ensemble_mean
X_test_for_error_ensemble = X_test.copy()
X_test_for_error_ensemble['mean_pred_oof'] = test_ensemble_mean
print("Error model data prepared successfully.")

# --- PART B: Train the XGBoost Error Model with pre-tuned parameters ---
# These are the best parameters you found from your Optuna run
best_params_error_xgb = {
    'eta': 0.010319058647945377,
    'max_depth': 10,
    'subsample': 0.9267794566303179,
    'colsample_bytree': 0.5015898655726251,
    'lambda': 0.0004339468292745306,
    'alpha': 0.001400353992325432
}

print("\n--- K-Fold training and saving XGBoost error model ---")
final_params_error_xgb = {**best_params_error_xgb, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1}
oof_error_preds_xgb = np.zeros(len(X))
test_error_preds_xgb = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_ensemble, grade_for_stratify)):
    print(f"  Training XGBoost Error Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_ensemble.iloc[train_idx], X_for_error_ensemble.iloc[val_idx]
    y_train, y_val = error_target_ensemble.iloc[train_idx], error_target_ensemble.iloc[val_idx]
    
    model = xgb.XGBRegressor(**final_params_error_xgb, n_estimators=3000, early_stopping_rounds=100)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_error_preds_xgb[val_idx] = model.predict(X_val)
    test_error_preds_xgb += model.predict(X_test_for_error_ensemble) / N_SPLITS

# --- PART C: Evaluate and Save Predictions ---
final_error_rmse_xgb = np.sqrt(mean_squared_error(error_target_ensemble, oof_error_preds_xgb))
print(f"\nXGBoost Error Model Final OOF RMSE: ${final_error_rmse_xgb:,.2f}")



--- Preparing data for error model training (using 3-model super-ensemble) ---
Error model data prepared successfully.

--- K-Fold training and saving XGBoost error model ---
  Training XGBoost Error Model - Fold 1/5...
  Training XGBoost Error Model - Fold 2/5...
  Training XGBoost Error Model - Fold 3/5...
  Training XGBoost Error Model - Fold 4/5...
  Training XGBoost Error Model - Fold 5/5...

XGBoost Error Model Final OOF RMSE: $61,822.06


NameError: name 'os' is not defined

In [8]:
import os

PREDS_SAVE_PATH = './model_predictions/'
os.makedirs(PREDS_SAVE_PATH, exist_ok=True)
np.save(f'{PREDS_SAVE_PATH}oof_error_preds_xgb.npy', oof_error_preds_xgb)
np.save(f'{PREDS_SAVE_PATH}test_error_preds_xgb.npy', test_error_preds_xgb)
print("XGBoost error model predictions saved successfully.")

XGBoost error model predictions saved successfully.


In [9]:
# =======================================================================================
#
# BLOCK 6: TRAIN & SAVE THE CHAMPION CATBOOST ERROR MODEL
#
# =======================================================================================
print("\n" + "="*70)

# --- PART A: Data is already prepared from the previous block ---
print("\n--- Using the same error target data for CatBoost training. ---")

# --- PART B: Train the CatBoost Error Model with pre-tuned parameters ---
# These are the best parameters you found from your Optuna run
best_params_error_cb = {
    'learning_rate': 0.020624018415943628,
    'depth': 9,
    'l2_leaf_reg': 0.004595377404698282,
    'subsample': 0.6631884536389594,
    'random_strength': 0.5195994030335256,
    'bagging_temperature': 0.10789200531811817
}

print("\n--- K-Fold training and saving CatBoost error model ---")
final_params_error_cb = {**best_params_error_cb, 'iterations': 3000, 'random_seed': RANDOM_STATE, 'verbose': 0, 'eval_metric': 'RMSE'}
oof_error_preds_cb = np.zeros(len(X))
test_error_preds_cb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_ensemble, grade_for_stratify)):
    print(f"  Training CatBoost Error Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_ensemble.iloc[train_idx], X_for_error_ensemble.iloc[val_idx]
    y_train, y_val = error_target_ensemble.iloc[train_idx], error_target_ensemble.iloc[val_idx]

    model = cb.CatBoostRegressor(**final_params_error_cb)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=0)
    
    oof_error_preds_cb[val_idx] = model.predict(X_val)
    test_error_preds_cb += model.predict(X_test_for_error_ensemble) / N_SPLITS

# --- PART C: Evaluate and Save Predictions ---
final_error_rmse_cb = np.sqrt(mean_squared_error(error_target_ensemble, oof_error_preds_cb))
print(f"\nCatBoost Error Model Final OOF RMSE: ${final_error_rmse_cb:,.2f}")

np.save(f'{PREDS_SAVE_PATH}oof_error_preds_cb.npy', oof_error_preds_cb)
np.save(f'{PREDS_SAVE_PATH}test_error_preds_cb.npy', test_error_preds_cb)
print("CatBoost error model predictions saved successfully.")



--- Using the same error target data for CatBoost training. ---

--- K-Fold training and saving CatBoost error model ---
  Training CatBoost Error Model - Fold 1/5...
  Training CatBoost Error Model - Fold 2/5...
  Training CatBoost Error Model - Fold 3/5...
  Training CatBoost Error Model - Fold 4/5...
  Training CatBoost Error Model - Fold 5/5...

CatBoost Error Model Final OOF RMSE: $61,719.37
CatBoost error model predictions saved successfully.
